# Experiment: SpectralBio Failure-Mode Screen (T4)

This notebook screens the existing SpectralBio panel and augmentation artifacts to identify a structured regime where likelihood-centric protein language model scoring appears to underweight local perturbation.

## Why this notebook exists
- The current paper is already strong as a benchmark and audit paper.
- To move beyond benchmark-only framing, we need a concrete failure mode candidate, not another generic AUC table.
- This notebook is deliberately cheap: it reuses panel-25 and ESM-1v augmentation artifacts that already exist.

## Success criteria
- Produce a ranked candidate table of interpretable regimes.
- Emit a single `ready_for_h100` decision with a shortlist for deeper validation.
- Package all outputs into a single zip for immediate download.


In [ ]:
from pathlib import Path

USE_GOOGLE_DRIVE = False
DRIVE_MOUNT_POINT = Path("/content/drive")
DRIVE_OUTPUT_SUBDIR = Path("MyDrive/SpectralBioFailureModeScreen")
SAVE_RESULTS_ZIP = True

if USE_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount(str(DRIVE_MOUNT_POINT))
    print(f"Drive mounted at {DRIVE_MOUNT_POINT}")
else:
    print("Drive mount skipped; outputs stay in the runtime unless OUTPUT_ROOT is changed later.")

print("ACABEI PODE IR PARA O PR?XIMO")


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = os.environ.get("SPECTRALBIO_REPO_URL", "https://github.com/DaviBonetto/SpectralBio.git")
REPO_BRANCH = os.environ.get("SPECTRALBIO_REPO_BRANCH", "codex/claw4s-rebuild")
DEFAULT_REPO_ROOT = Path("/content/Stanford-Claw4s")
ENV_REPO_ROOT = os.environ.get("SPECTRALBIO_REPO_ROOT", "").strip()

def _looks_like_repo(path: Path) -> bool:
    return (path / "src" / "spectralbio").exists() and (path / "notebooks").exists()

candidate_roots = []
if ENV_REPO_ROOT:
    candidate_roots.append(Path(ENV_REPO_ROOT).expanduser())
candidate_roots.extend([Path.cwd(), Path.cwd().parent, DEFAULT_REPO_ROOT])
REPO_ROOT = next((path.resolve() for path in candidate_roots if _looks_like_repo(path)), DEFAULT_REPO_ROOT)
BOOTSTRAP_MARKER = REPO_ROOT / ".colab_bootstrap_failure_mode_screen_complete"

if not _looks_like_repo(REPO_ROOT):
    REPO_ROOT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(REPO_ROOT)])

os.chdir(REPO_ROOT)
subprocess.run(["git", "fetch", "origin", REPO_BRANCH], check=False)
subprocess.run(["git", "checkout", REPO_BRANCH], check=False)

src_path = REPO_ROOT / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

if not BOOTSTRAP_MARKER.exists():
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", ".", "--no-deps"])
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "numpy==2.1.3",
            "pandas==2.2.3",
            "matplotlib==3.9.2",
            "scipy==1.14.1",
            "scikit-learn==1.5.2",
        ]
    )
    BOOTSTRAP_MARKER.write_text("ok\n", encoding="utf-8")
    print("Dependencies installed without restarting the runtime.")
else:
    print("Bootstrap marker found; skipping reinstall.")

print("REPO_ROOT =", REPO_ROOT)
print("Python =", sys.version.split()[0])
print("ACABEI PODE IR PARA O PR?XIMO")


## Plan

1. Locate the frozen panel-25 and augmentation artifacts that already exist.
2. Build a feature-enriched variant table from the reference panel and the available ESM-1v audits.
3. Rank candidate regimes by rescue strength, benign harm, and support outside the flagship gene.
4. Emit a single shortlist plus a validation pool sized for one careful H100 run.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import FileLink, display

from spectralbio.utils.io import ensure_dir

OUTPUT_ROOT = (
    DRIVE_MOUNT_POINT / DRIVE_OUTPUT_SUBDIR
    if USE_GOOGLE_DRIVE
    else REPO_ROOT / "supplementary" / "failure_mode_screen_t4"
)
ARTIFACT_ROOT = OUTPUT_ROOT / "failure_mode_screen"
TABLES_DIR = ARTIFACT_ROOT / "tables"
FIGURES_DIR = ARTIFACT_ROOT / "figures"

for directory in (ARTIFACT_ROOT, TABLES_DIR, FIGURES_DIR):
    ensure_dir(directory)

RESCUE_THRESHOLD = 0.12
HARM_THRESHOLD = 0.12
MAX_AUG_CANDIDATES = 8
MAX_REFERENCE_EXTENSION = 6
MAX_BENIGN_REGIME = 4
MIN_RESCUED_POSITIVES = 4
MIN_REFERENCE_GAP = 0.05

POSITIVE_AA = set("KRH")
NEGATIVE_AA = set("DE")
HYDROPHOBIC_AA = set("AVLIMFWY")
AROMATIC_AA = set("FWYH")
SPECIAL_AA = set("GPC")

def resolve_existing_path(raw: str | Path) -> Path:
    raw_path = Path(raw)
    if raw_path.exists():
        return raw_path
    raw_text = str(raw).replace("\\", "/")
    for prefix in (
        "/content/Stanford-Claw4s/",
        "/teamspace/studios/this_studio/Stanford-Claw4s/",
    ):
        if raw_text.startswith(prefix):
            candidate = REPO_ROOT / raw_text[len(prefix):]
            if candidate.exists():
                return candidate
    return raw_path

def minmax_normalize(series: pd.Series) -> pd.Series:
    values = series.astype(float)
    minimum = float(values.min())
    maximum = float(values.max())
    if maximum <= minimum:
        return pd.Series(np.zeros(len(values), dtype=float), index=series.index)
    return (values - minimum) / (maximum - minimum)

def charge_group(aa: str) -> str:
    aa = str(aa).upper()
    if aa in POSITIVE_AA:
        return "positive"
    if aa in NEGATIVE_AA:
        return "negative"
    return "neutral"

def polarity_group(aa: str) -> str:
    aa = str(aa).upper()
    if aa in HYDROPHOBIC_AA:
        return "hydrophobic"
    return "polar_or_special"

def pick_best(paths: list[Path], gene_hint: str | None = None) -> Path | None:
    scored = []
    for path in paths:
        text = str(path).lower()
        score = 0
        if "results-msh2" in text:
            score += 10
        if "final_accept_part3" in text:
            score += 5
        if "panel25" in text:
            score += 5
        if gene_hint and gene_hint.lower() in text:
            score += 2
        scored.append((score, len(text), path))
    if not scored:
        return None
    scored.sort(reverse=True)
    return scored[0][2]

panel_manifest_candidates = [
    REPO_ROOT / "supplementary" / "final_accept_a100" / "panel25" / "support_ranked_panel_manifest.json",
]
panel_manifest_candidates += sorted(REPO_ROOT.glob("notebooks/**/support_ranked_panel_manifest.json"))
panel_manifest_candidates += sorted(REPO_ROOT.glob("supplementary/**/support_ranked_panel_manifest.json"))
panel_manifest_path = None
for candidate in panel_manifest_candidates:
    resolved = resolve_existing_path(candidate)
    if resolved.exists():
        panel_manifest_path = resolved
        break
if panel_manifest_path is None:
    raise FileNotFoundError("Could not find support_ranked_panel_manifest.json anywhere under the repo.")

panel_root = panel_manifest_path.parent
multigene_dir = panel_root / "multigene"
multigene_summary_path = multigene_dir / "multigene_summary.csv"
support_gene_table_path = multigene_dir / "support_ranked_gene_table.csv"

augmentation_table_candidates = sorted(REPO_ROOT.glob("notebooks/**/augmentation_table.csv"))
augmentation_table_candidates += sorted(REPO_ROOT.glob("supplementary/**/augmentation_table.csv"))
augmentation_table_map = {}
for path in augmentation_table_candidates:
    gene = path.parent.name.upper()
    chosen = augmentation_table_map.get(gene)
    if chosen is None:
        augmentation_table_map[gene] = path
        continue
    augmentation_table_map[gene] = pick_best([chosen, path], gene_hint=gene)

print("panel_manifest_path =", panel_manifest_path)
print("multigene_dir =", multigene_dir)
print("available augmentation genes =", sorted(augmentation_table_map))
print("ACABEI PODE IR PARA O PR?XIMO")


In [ ]:
panel_manifest = json.loads(panel_manifest_path.read_text(encoding="utf-8"))
multigene_summary = pd.read_csv(multigene_summary_path)
support_gene_table = pd.read_csv(support_gene_table_path)

manifest_rows = []
for gene, payload in panel_manifest["genes"].items():
    sequence_path = resolve_existing_path(payload["sequence_path"])
    sequence = "".join(
        line.strip() for line in sequence_path.read_text(encoding="utf-8").splitlines() if line and not line.startswith(">")
    )
    manifest_rows.append(
        {
            "gene": gene.upper(),
            "sequence_length": len(sequence),
            "support_rank": int(payload.get("support_rank", 0)),
        }
    )
manifest_df = pd.DataFrame(manifest_rows)

reference_frames = []
for gene_dir in sorted(multigene_dir.iterdir()):
    if not gene_dir.is_dir():
        continue
    score_candidates = sorted(gene_dir.glob("*_facebook_esm2_t30_150M_UR50D_scores.csv"))
    if not score_candidates:
        continue
    frame = pd.read_csv(score_candidates[0])
    frame["source_score_path"] = str(score_candidates[0])
    reference_frames.append(frame)
reference_df = pd.concat(reference_frames, ignore_index=True)
reference_df["gene"] = reference_df["gene"].str.upper()
reference_df = reference_df.merge(manifest_df, on="gene", how="left")
reference_df["ll_norm"] = reference_df.groupby("gene")["ll_proper"].transform(minmax_normalize)
reference_df["frob_norm"] = reference_df.groupby("gene")["frob_dist"].transform(minmax_normalize)
reference_df["pair_norm"] = (0.55 * reference_df["frob_norm"]) + (0.45 * reference_df["ll_norm"])
reference_df["reference_advantage"] = reference_df["pair_norm"] - reference_df["ll_norm"]
reference_df["substitution"] = reference_df["wt_aa"].astype(str) + ">" + reference_df["mut_aa"].astype(str)
reference_df["position_1based"] = reference_df["position"].astype(int) + 1
reference_df["available_in_aug"] = reference_df["gene"].isin(augmentation_table_map)

aug_frames = []
for gene, table_path in sorted(augmentation_table_map.items()):
    frame = pd.read_csv(table_path)
    frame["gene"] = frame["gene"].str.upper()
    frame["source_augmentation_path"] = str(table_path)
    aug_frames.append(frame)
aug_df = pd.concat(aug_frames, ignore_index=True)
aug_df = aug_df.merge(manifest_df, on="gene", how="left")
aug_df["substitution"] = aug_df["wt_aa"].astype(str) + ">" + aug_df["mut_aa"].astype(str)
aug_df["position_1based"] = aug_df["position"].astype(int) + 1
aug_df["rescue_margin"] = aug_df["augmented_pair_fixed_055"] - aug_df["esm1v_ll_mean_norm"]
aug_df["reference_advantage"] = aug_df["reference_pair_norm"] - aug_df["reference_ll_norm"]

print("reference variants =", len(reference_df))
print("augmentation variants =", len(aug_df))
print("ACABEI PODE IR PARA O PR?XIMO")


In [ ]:
def add_feature_columns(frame: pd.DataFrame) -> pd.DataFrame:
    data = frame.copy()
    data["to_proline"] = data["mut_aa"].astype(str).str.upper().eq("P")
    data["from_glycine"] = data["wt_aa"].astype(str).str.upper().eq("G")
    data["to_cysteine"] = data["mut_aa"].astype(str).str.upper().eq("C")
    data["involves_special_residue"] = (
        data["wt_aa"].astype(str).str.upper().isin(SPECIAL_AA)
        | data["mut_aa"].astype(str).str.upper().isin(SPECIAL_AA)
    )
    data["aromatic_change"] = (
        data["wt_aa"].astype(str).str.upper().isin(AROMATIC_AA)
        | data["mut_aa"].astype(str).str.upper().isin(AROMATIC_AA)
    )
    data["charge_flip"] = data["wt_aa"].map(charge_group) != data["mut_aa"].map(charge_group)
    data["polarity_flip"] = data["wt_aa"].map(polarity_group) != data["mut_aa"].map(polarity_group)
    data["n_term_30"] = data["position_1based"] <= 30
    data["n_term_50"] = data["position_1based"] <= 50
    data["c_term_50"] = (data["sequence_length"] - data["position_1based"]) < 50
    data["high_frob_low_ll"] = (data["frob_norm"] >= 0.75) & (data["ll_norm"] <= 0.50)
    data["very_high_disagreement"] = (data["frob_norm"] - data["ll_norm"]) >= 0.35
    data["high_frob_low_ll_and_special"] = data["high_frob_low_ll"] & data["involves_special_residue"]
    data["high_frob_low_ll_and_to_proline"] = data["high_frob_low_ll"] & data["to_proline"]
    data["high_frob_low_ll_and_from_glycine"] = data["high_frob_low_ll"] & data["from_glycine"]
    data["high_frob_low_ll_and_to_cysteine"] = data["high_frob_low_ll"] & data["to_cysteine"]
    return data

reference_df = add_feature_columns(reference_df)
aug_df = add_feature_columns(aug_df)

regime_columns = [
    "to_proline",
    "from_glycine",
    "to_cysteine",
    "involves_special_residue",
    "aromatic_change",
    "charge_flip",
    "polarity_flip",
    "n_term_30",
    "n_term_50",
    "c_term_50",
    "high_frob_low_ll",
    "very_high_disagreement",
    "high_frob_low_ll_and_special",
    "high_frob_low_ll_and_to_proline",
    "high_frob_low_ll_and_from_glycine",
    "high_frob_low_ll_and_to_cysteine",
]

def summarize_regime(column: str) -> dict:
    aug_subset = aug_df[aug_df[column]].copy()
    ref_subset = reference_df[reference_df[column]].copy()
    path_subset = aug_subset[aug_subset["label"] == 1]
    benign_subset = aug_subset[aug_subset["label"] == 0]
    rescued_positive = path_subset[path_subset["rescue_margin"] >= RESCUE_THRESHOLD]
    harmed_benign = benign_subset[benign_subset["rescue_margin"] >= HARM_THRESHOLD]
    ref_path = ref_subset[ref_subset["label"] == 1]
    ref_benign = ref_subset[ref_subset["label"] == 0]
    path_margin = float(path_subset["rescue_margin"].mean()) if not path_subset.empty else 0.0
    benign_margin = float(benign_subset["rescue_margin"].mean()) if not benign_subset.empty else 0.0
    reference_path_adv = float(ref_path["reference_advantage"].mean()) if not ref_path.empty else 0.0
    reference_benign_adv = float(ref_benign["reference_advantage"].mean()) if not ref_benign.empty else 0.0
    reference_gap = reference_path_adv - reference_benign_adv
    genes_with_rescue = int(rescued_positive["gene"].nunique()) if not rescued_positive.empty else 0
    composite = (
        rescued_positive.shape[0]
        * (1.0 + (genes_with_rescue / 4.0))
        * max((path_margin - benign_margin) + max(reference_gap, 0.0) + 0.05, 0.0)
    ) / (1.0 + harmed_benign.shape[0])
    return {
        "regime": column,
        "rescued_positive_count": int(rescued_positive.shape[0]),
        "harmed_benign_count": int(harmed_benign.shape[0]),
        "genes_with_rescued_positive": genes_with_rescue,
        "path_rescue_margin_mean": path_margin,
        "benign_rescue_margin_mean": benign_margin,
        "reference_gap": reference_gap,
        "composite_score": float(composite),
    }

candidate_failure_modes = pd.DataFrame([summarize_regime(column) for column in regime_columns])
candidate_failure_modes = candidate_failure_modes.sort_values(
    ["composite_score", "rescued_positive_count", "reference_gap"],
    ascending=[False, False, False],
).reset_index(drop=True)

selected_row = None
for _, row in candidate_failure_modes.iterrows():
    if int(row["rescued_positive_count"]) >= MIN_RESCUED_POSITIVES and float(row["reference_gap"]) >= MIN_REFERENCE_GAP:
        selected_row = row.to_dict()
        break
if selected_row is None:
    selected_row = candidate_failure_modes.iloc[0].to_dict()
    ready_for_h100 = False
    selection_reason = "No regime passed the strict gate; fallback regime kept for manual inspection only."
else:
    ready_for_h100 = True
    selection_reason = "Selected regime passed the minimum rescued-positive and reference-gap thresholds."

selected_regime = str(selected_row["regime"])
aug_df["selected_regime"] = aug_df[selected_regime].fillna(False).astype(bool)
reference_df["selected_regime"] = reference_df[selected_regime].fillna(False).astype(bool)

print("selected_regime =", selected_regime)
print("ready_for_h100 =", ready_for_h100)
print(candidate_failure_modes.head(8))
print("ACABEI PODE IR PARA O PR?XIMO")


In [ ]:
def build_validation_pool() -> pd.DataFrame:
    selected_aug = aug_df[aug_df["selected_regime"]].copy()
    selected_reference = reference_df[reference_df["selected_regime"]].copy()
    aug_candidates = (
        selected_aug[selected_aug["label"] == 1]
        .sort_values(["rescue_margin", "reference_advantage"], ascending=False)
        .head(MAX_AUG_CANDIDATES)
        .copy()
    )
    aug_candidates["validation_role"] = "candidate_positive_existing_aug"
    reference_extension = (
        selected_reference[(selected_reference["label"] == 1) & (~selected_reference["available_in_aug"])]
        .sort_values(["reference_advantage", "frob_norm"], ascending=False)
        .head(MAX_REFERENCE_EXTENSION)
        .copy()
    )
    reference_extension["validation_role"] = "candidate_positive_reference_extension"
    regime_benign = (
        selected_aug[selected_aug["label"] == 0]
        .sort_values(["rescue_margin", "reference_advantage"], ascending=False)
        .head(MAX_BENIGN_REGIME)
        .copy()
    )
    regime_benign["validation_role"] = "regime_benign"
    controls = []
    used = set()
    for _, candidate in pd.concat([aug_candidates, reference_extension], ignore_index=True).iterrows():
        pool = reference_df[
            (reference_df["gene"] == candidate["gene"])
            & (reference_df["label"] == 1)
            & (~reference_df["selected_regime"])
            & (~reference_df["name"].isin(used))
        ].copy()
        if pool.empty:
            continue
        pool["match_distance"] = (pool["position"].astype(int) - int(candidate["position"])).abs() + (25.0 * (pool["ll_norm"] - float(candidate.get("ll_norm", 0.0))).abs())
        picked = pool.sort_values(["match_distance", "reference_advantage"]).iloc[0].copy()
        used.add(str(picked["name"]))
        picked["validation_role"] = "matched_positive_control"
        controls.append(picked)
    matched_controls = pd.DataFrame(controls)
    validation_pool = pd.concat([aug_candidates, reference_extension, matched_controls, regime_benign], ignore_index=True, sort=False)
    keep_columns = [
        "gene",
        "name",
        "position",
        "wt_aa",
        "mut_aa",
        "label",
        "validation_role",
        "selected_regime",
        "substitution",
        "sequence_length",
        "support_rank",
        "ll_norm",
        "frob_norm",
        "pair_norm",
        "reference_advantage",
        "esm1v_ll_mean_norm",
        "augmented_pair_fixed_055",
        "rescue_margin",
    ]
    for column in keep_columns:
        if column not in validation_pool.columns:
            validation_pool[column] = np.nan
    return validation_pool[keep_columns].sort_values(["validation_role", "gene", "position", "mut_aa", "name"]).reset_index(drop=True)

validation_pool = build_validation_pool()
summary_payload = {
    "screen_name": "SpectralBio failure-mode screen",
    "selected_regime": selected_regime,
    "ready_for_h100": bool(ready_for_h100),
    "selection_reason": selection_reason,
    "thresholds": {
        "rescue_threshold": RESCUE_THRESHOLD,
        "harm_threshold": HARM_THRESHOLD,
        "min_rescued_positives": MIN_RESCUED_POSITIVES,
        "min_reference_gap": MIN_REFERENCE_GAP,
    },
    "selected_regime_summary": selected_row,
    "panel_manifest_path": str(panel_manifest_path),
    "validation_pool_counts": validation_pool["validation_role"].value_counts().to_dict(),
    "recommended_h100_variant_count": int(validation_pool.shape[0]),
}
candidate_failure_modes.to_csv(TABLES_DIR / "candidate_failure_modes.csv", index=False)
validation_pool.to_csv(TABLES_DIR / "candidate_validation_pool.csv", index=False)
(TABLES_DIR / "candidate_failure_mode_summary.json").write_text(json.dumps(summary_payload, indent=2, ensure_ascii=False), encoding="utf-8")

top_plot = candidate_failure_modes.head(10).copy()
plt.figure(figsize=(10, 5))
plt.barh(top_plot["regime"][::-1], top_plot["composite_score"][::-1], color="#2459a6")
plt.xlabel("Composite candidate score")
plt.title("Top candidate failure modes")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "candidate_failure_modes_top10.png", dpi=180, bbox_inches="tight")
plt.close()

print(validation_pool.head(20))
print("ACABEI PODE IR PARA O PR?XIMO")


In [ ]:
import shutil

ZIP_BASE = ARTIFACT_ROOT / "failure_mode_screen_bundle"
zip_path = Path(shutil.make_archive(str(ZIP_BASE), "zip", ARTIFACT_ROOT))
print("Bundle written to", zip_path)
display(FileLink(str(zip_path)))

try:
    from google.colab import files

    files.download(str(zip_path))
    print("Auto-download requested through google.colab.files.download.")
except Exception as error:
    print("Auto-download was not triggered:", error)

print("ACABEI PODE IR PARA O PR?XIMO")
